> **Note:** The primary objective of this notebook is to understand and demonstrate the working of **ColumnTransformer**.

>  Therefore, the focus is on applying different preprocessing techniques to different feature columns using `ColumnTransformer`.

> Advanced preprocessing strategies, feature engineering, hyperparameter tuning, and model optimization are intentionally kept out of scope and can be explored separately.


# ***Step 1: Import Libraries and load dataset***

In [110]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [111]:
df = pd.read_excel('ML_Preprocessing_Practice_Dataset.excel')

In [112]:
df.head()

,Education_Level,City,Department,Employment_Type,Age,Monthly_Income,Performance
0,High School,Hyderabad,Marketing,Part-Time,27.0,21250.0,Average
1,Bachelor,Chennai,Sales,Part-Time,24.0,45231.0,Average
2,High School,Chennai,Sales,Contract,36.0,25366.0,Average
3,High School,Bengaluru,IT,Contract,45.0,22289.0,Average
4,High School,Bengaluru,IT,Full-Time,26.0,24658.0,Average


## ***Step 2: Check Missing Values***

In [113]:
df.isnull().sum()

Education_Level     6
City               13
Department         11
Employment_Type     8
Age                15
Monthly_Income      7
Performance         0
dtype: int64

In [114]:
num_cols = ['Age', 'Monthly_Income']

df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

In [115]:
df['Education_Level'].value_counts()

Education_Level
Bachelor       1948
Master         1080
Diploma        1060
High School     841
PhD             595
Name: count, dtype: int64

In [116]:
cat_cols = [
    'Education_Level',
    'City',
    'Department',
    'Employment_Type',
    'Performance'
]

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [117]:
df.isnull().sum()

Education_Level    0
City               0
Department         0
Employment_Type    0
Age                0
Monthly_Income     0
Performance        0
dtype: int64

## ***Step 3: Separate Features and Target***

In [118]:
X = df.drop('Performance', axis=1)
y = df['Performance']

## ***Note:🔴***
> LabelEncoder is applied separately because it is used to encode the target variable (y).

> ColumnTransformer is designed to preprocess only the input features (X), not the target.

> Therefore, the target column should not be included in ColumnTransformer.

In [119]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

## ***Step 4: Identify Numerical and Categorical Columns***

In [120]:
num_cols = ['Age', 'Monthly_Income']

cat_cols = [
    'City',
    'Department',
    'Employment_Type'
]

In [129]:
df['Employment_Type'].value_counts()

Employment_Type
Part-Time    1876
Full-Time    1859
Contract     1795
Name: count, dtype: int64

In [121]:
from sklearn.preprocessing import OrdinalEncoder

education_order = [[
    'High School',
    'Diploma',
    'Bachelor',
    'Master',
    'PhD'
]]

ordinal_encoder = OrdinalEncoder(categories=education_order)

## ***Step 5: Create ColumnTransformer***

- We'll:

> Apply StandardScaler to numerical columns.

> Apply OneHotEncoder to categorical columns.

In [122]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

In [135]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore',drop='first'), cat_cols),  # handle_unknown='ignore' → Ignores unseen categories during prediction instead of throwing an error.
        ('edu', ordinal_encoder, ['Education_Level']),
    ]
)

## ***Step 6: Train-Test Split***

In [136]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [137]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [138]:
# feature_names = preprocessor.get_feature_names_out()

In [140]:
# X_transformed_df = pd.DataFrame(
#     X_train_transformed,
#     columns=feature_names)
# X_transformed_df

```
Raw Data
    │
    ▼
Train-Test Split
    │
    ├──────────────┐
    │              │
X_train         X_test
    │              │
fit_transform() transform()
    │              │
    └──────┬───────┘
           ▼
Preprocessed Data
```

In [84]:
print(cat_cols)

['City', 'Department', 'Employment_Type']


## ***Step 7: Import the Model***

In [85]:
from sklearn.linear_model import LogisticRegression

##  ***Step 8: Create the Model***

In [86]:
model = LogisticRegression(random_state=42)

# LogisticRegression() → Creates the classifier.
# random_state=42 → Ensures reproducible results.

## ***Step 9: Train the Model***

In [87]:
model.fit(X_train_transformed, y_train)

LogisticRegression(random_state=42)

## ***Step 10: Make Predictions***

In [88]:
y_pred = model.predict(X_test_transformed)

In [89]:
print(y_pred)

[2 2 0 ... 0 1 0]


## ***Step 11: Evaluate the Model***

In [90]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9972875226039783


### ***Confusion Matrix***

In [91]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[639   0   0]
 [  0 167   2]
 [  0   1 297]]


### ***Classification Report***

In [92]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       639
           1       0.99      0.99      0.99       169
           2       0.99      1.00      0.99       298

    accuracy                           1.00      1106
   macro avg       1.00      0.99      1.00      1106
weighted avg       1.00      1.00      1.00      1106



In [93]:
# Convert numeric labels back to original class names
y_test_actual = le.inverse_transform(y_test)
y_pred_actual = le.inverse_transform(y_pred)
import pandas as pd

result = pd.DataFrame({
    'Actual': y_test_actual,
    'Predicted': y_pred_actual
})

print(result)

         Actual  Predicted
0          Good       Good
1          Good       Good
2       Average    Average
3          Good       Good
4     Excellent  Excellent
...         ...        ...
1101    Average    Average
1102       Good       Good
1103    Average    Average
1104  Excellent  Excellent
1105    Average    Average

[1106 rows x 2 columns]


In [94]:
result['Status'] = ['Correct' if a == p else 'Wrong' for a, p in zip(y_test_actual, y_pred_actual)]
result['Match'] = result['Actual'] == result['Predicted']   # True/False

# Summary
print(result['Status'].value_counts())
print("\nAccuracy:", result['Match'].mean()*100, "%")

Status
Correct    1103
Wrong         3
Name: count, dtype: int64

Accuracy: 99.72875226039784 %


In [95]:
result

,Actual,Predicted,Status,Match
0,Good,Good,Correct,True
1,Good,Good,Correct,True
2,Average,Average,Correct,True
3,Good,Good,Correct,True
4,Excellent,Excellent,Correct,True
...,...,...,...,...
1101,Average,Average,Correct,True
1102,Good,Good,Correct,True
1103,Average,Average,Correct,True
1104,Excellent,Excellent,Correct,True
